In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import kurtosis, spearmanr
import itertools
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. PARAMETRIC CONFIGURATION
# ==============================================================================

# 1. Get the directory where the current Jupyter Notebook is located
# Path.cwd() gets the current working directory of the notebook
notebook_dir = Path.cwd() 

# 2. Navigate up to the parent directory (crypto-perpetual-arbitrage)
project_dir = notebook_dir.parent 

# 3. Navigate into the 'data' folder
DATA_DIR = project_dir / "data"

# 4. Point to the specific parquet file
MASTER_PARQUET = DATA_DIR / "BTC_TAGGED_MASTER.parquet"

# (Optional) Verify the paths are correct
print(f"Data Directory: {DATA_DIR}")
print(f"Master Parquet: {MASTER_PARQUET}")

GLOBAL_START = pd.Timestamp('2025-03-01 00:00:00')
GLOBAL_END = pd.Timestamp('2025-12-31 23:59:00')

ROLLING_WINDOW_MINS = 1440  # 24-Hour Ex-Ante Lookback
MIN_VALID_MINS = 720        # Require at least 12h of clean data inside the 24h window
ADJ_SPREAD_PREFIX = 'spread_padj_idx_mid_avg'

# 1. Taker Fees from VIP Table (in bps)
TAKER_FEES = {
    'binf': 1.70, 'byde': 3.00, 'okex': 1.75, 
    'kusw': 2.50, 'gate': 2.00, 'hype': 1.44
}
ASSUMED_SLIPPAGE_BPS = 0.5 # Per leg

# ==============================================================================
# PART 1: MACRO STYLIZED FACTS (MAB & ROLLING RELIABILITY)
# ==============================================================================
def run_part1_stylized_facts(df, all_pairs):
    print("\n" + "="*125)
    print(" 📘 PART 1: MACRO STYLIZED FACTS & MINIMUM ARBITRAGE BANDS (MAB)")
    print(" ⚙️  APPROACH: STATIC EX-POST WITH DAILY ROLLING SAMPLING")
    print("="*125)
    print(" 📖 METRICS GLOSSARY:")
    print(" - StdDev(bps)    : The unconditional volatility of the spread.")
    print(" - MAB(bps)       : Minimum Arbitrage Band. (Round-Trip Taker Fees + Assumed 2.0 bps Slippage).")
    print(" - Req. Sigma     : (MAB / StdDev). The theoretical Z-score magnitude required just to break even.")
    print(" - Static ADF     : Full-year Dickey-Fuller test (proves global cointegration).")
    print(" - Roll Stat %    : Percentage of rolling 24h windows that passed the ADF test (proves continuous tradability).")
    print("="*125)

    results = []
    for exch_A, exch_B in all_pairs:
        pair_col = f"{ADJ_SPREAD_PREFIX}_{exch_A}_vs_{exch_B}"
        if pair_col not in df.columns: continue
            
        stale_A = df.get(f'stale_{exch_A}', pd.Series(True, index=df.index))
        stale_B = df.get(f'stale_{exch_B}', pd.Series(True, index=df.index))
        active_mask = (~stale_A) & (~stale_B)
        
        y_continuous = df[pair_col].where(active_mask)
        x_continuous = y_continuous.shift(1)
        acf1 = y_continuous.corr(x_continuous)
        
        clean_series = y_continuous.dropna()
        if len(clean_series) < 10000: continue
        
        pair_name = f"{exch_A.split('-')[0].upper()[:4]}/{exch_B.split('-')[0].upper()[:4]}"
        
        # 1. Static Global ADF
        adf_series = clean_series.iloc[::10] # Sampled for speed
        _, p_value_static, _, _, _, _ = adfuller(adf_series, maxlag=5)
        
        # 2. Rolling Daily ADF Reliability
        stationary_days, total_tested_days = 0, 0
        sample_points = range(1440, len(clean_series), 1440)
        for pt in sample_points:
            window = clean_series.iloc[pt-1440 : pt]
            if len(window) > 1000:
                try:
                    _, pval_roll, _, _, _, _ = adfuller(window, maxlag=1)
                    if pval_roll < 0.05: stationary_days += 1
                    total_tested_days += 1
                except: continue
        rolling_stat_pct = (stationary_days / total_tested_days * 100) if total_tested_days > 0 else 0
        
        # 3. MAB & Required Sigma
        fee_A = TAKER_FEES.get(exch_A, 2.0) 
        fee_B = TAKER_FEES.get(exch_B, 2.0)
        total_mab = ((fee_A + fee_B) * 2) + (ASSUMED_SLIPPAGE_BPS * 4)
        std_dev = clean_series.std()
        req_sigma = total_mab / std_dev if std_dev > 0 else np.nan

        results.append({
            'Pair': pair_name, 
            'Mean': clean_series.mean(), 
            'StdDev': std_dev,
            'MAB(bps)': total_mab,
            'Req. Sigma': req_sigma,
            'ACF(1)': acf1, 
            'Static ADF': p_value_static,
            'Roll Stat %': rolling_stat_pct, 
            'Active Days': len(clean_series) / 1440.0
        })

    res_df = pd.DataFrame(results).sort_values('StdDev')
    fmt = {'Mean': "{:.3f}", 'StdDev': "{:.3f}", 'MAB(bps)': "{:.2f}", 'Req. Sigma': "{:.2f}",
           'ACF(1)': "{:.4f}", 'Static ADF': "{:.4f}", 'Roll Stat %': "{:.1f}%", 'Active Days': "{:.1f}"}
    for col, f in fmt.items(): 
        if col in res_df.columns: res_df[col] = res_df[col].apply(lambda x: f.format(x) if pd.notna(x) else "N/A")
    print(res_df.to_string(index=False))

# ==============================================================================
# PART 2: TACTICAL SIGNAL FREQUENCY 
# ==============================================================================
def run_part2_signal_frequency(df, all_pairs):
    print("\n" + "="*125)
    print(" 📘 PART 2: TACTICAL SIGNAL FREQUENCY")
    print(" ⚙️  APPROACH: DYNAMIC ROLLING EX-ANTE (1440m WINDOW)")
    print("="*125)
    print(" 📖 METRICS GLOSSARY:")
    print(" - Naive Z > 3σ   : Breaches of a simple rolling standard deviation.")
    print(" - OU S-Score     : Breaches of the AR(1) filtered Ornstein-Uhlenbeck stochastic equilibrium.")
    print(" - Filter Toxicity: The percentage of Naive signals that the OU model correctly identifies as macro-drift (false positives).")
    print("="*125)

    results = []
    for exch_A, exch_B in all_pairs:
        pair_col = f"{ADJ_SPREAD_PREFIX}_{exch_A}_vs_{exch_B}"
        if pair_col not in df.columns: continue
            
        stale_A = df.get(f'stale_{exch_A}', pd.Series(True, index=df.index))
        stale_B = df.get(f'stale_{exch_B}', pd.Series(True, index=df.index))
        active_mask = (~stale_A) & (~stale_B)
        pair_name = f"{exch_A.split('-')[0].upper()[:4]}/{exch_B.split('-')[0].upper()[:4]}"

        y = df[pair_col].where(active_mask)
        x = y.shift(1)
        
        roll_mean = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).mean()
        roll_std = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).std()
        roll_cov = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).cov(x)
        roll_var_x = x.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).var()
        roll_var_y = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).var()
        
        rho_1 = (roll_cov / roll_var_x).clip(lower=1e-5, upper=0.999)
        m = (roll_mean - rho_1 * x.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).mean()) / (1 - rho_1)
        var_e = (roll_var_y - (rho_1**2) * roll_var_x).clip(lower=1e-8)
        sigma_eq = np.sqrt(var_e / (1 - rho_1**2))
        
        z_scores = ((y - roll_mean) / roll_std).dropna()
        s_scores = ((y - m) / sigma_eq).dropna()
        
        if len(s_scores) < 10000: continue
        z_3s, s_3s = (np.abs(z_scores) > 3.0).sum(), (np.abs(s_scores) > 3.0).sum()
        
        results.append({
            'Pair': pair_name, 'Naive Z > 3σ': z_3s, 'OU S-Score > 3σ': s_3s,
            'Naive Z > 4σ': (np.abs(z_scores) > 4.0).sum(), 'OU S-Score > 4σ': (np.abs(s_scores) > 4.0).sum(),
            'Filtered Toxicity %': (1 - (s_3s / max(z_3s, 1))) * 100
        })

    res_df = pd.DataFrame(results).sort_values('OU S-Score > 3σ', ascending=False)
    res_df['Filtered Toxicity %'] = res_df['Filtered Toxicity %'].apply(lambda x: f"{x:.1f}%")
    print(res_df.to_string(index=False))

# ==============================================================================
# PART 3: OU MODEL VALIDATION
# ==============================================================================
def run_part3_ou_validation(df, all_pairs):
    print("\n" + "="*125)
    print(" 📘 PART 3: OU STOCHASTIC MODEL VALIDATION & RESIDUAL DIAGNOSTICS")
    print(" ⚙️  APPROACH: STATIC EX-POST (FULL SAMPLE REGRESSION)")
    print("="*125)
    print(" 📖 METRICS GLOSSARY:")
    print(" - AR(1) Beta   : The mean-reversion speed. Closer to 1.0 = slower decay.")
    print(" - Mean Delta   : Absolute difference between Theoretical limits and Empirical physical order book centers.")
    print("                  Values near 0 prove the model is perfectly anchored to reality.")
    print("="*125)

    results = []
    for exch_A, exch_B in all_pairs:
        pair_col = f"{ADJ_SPREAD_PREFIX}_{exch_A}_vs_{exch_B}"
        if pair_col not in df.columns: continue
            
        stale_A = df.get(f'stale_{exch_A}', pd.Series(True, index=df.index))
        stale_B = df.get(f'stale_{exch_B}', pd.Series(True, index=df.index))
        active_mask = (~stale_A) & (~stale_B)
        pair_name = f"{exch_A.split('-')[0].upper()[:4]}/{exch_B.split('-')[0].upper()[:4]}"

        y = df[pair_col].where(active_mask)
        x = y.shift(1)
        df_reg = pd.DataFrame({'y': y, 'x': x}).dropna()
        if len(df_reg) < 10000: continue

        model = sm.OLS(df_reg['y'], sm.add_constant(df_reg['x'])).fit()
        alpha, beta = model.params.iloc[0], model.params.iloc[1]
        
        if beta >= 1 or beta <= 0: continue
        theo_mean = alpha / (1 - beta)
        emp_mean = df_reg['y'].mean()
        lb_pval = acorr_ljungbox(model.resid, lags=[10], return_df=True)['lb_pvalue'].iloc[0]
        
        results.append({
            'Pair': pair_name, 'AR(1) Beta': beta, 'Theo Mean(μ)': theo_mean,
            'Emp Mean(x̄)': emp_mean, 'Mean Delta(Δ)': abs(theo_mean - emp_mean), 'Ljung-Box': lb_pval
        })

    res_df = pd.DataFrame(results).sort_values('AR(1) Beta')
    fmt = {'AR(1) Beta': "{:.4f}", 'Theo Mean(μ)': "{:.3f}", 'Emp Mean(x̄)': "{:.3f}", 
           'Mean Delta(Δ)': "{:.4f}", 'Ljung-Box': "{:.4f}"}
    for col, f in fmt.items(): res_df[col] = res_df[col].apply(f.format)
    print(res_df.to_string(index=False))

# ==============================================================================
# PART 4: MODEL USEFULNESS & RANK CORRELATION
# ==============================================================================
def calculate_anchored_empirical_ttr(y_series, m_series, s_scores, sigma_thresh):
    triggers = np.where(np.abs(s_scores) >= sigma_thresh)[0]
    clean_triggers = []
    last_t = -60
    for t in triggers:
        if t - last_t >= 60:
            clean_triggers.append(t)
            last_t = t
            
    if not clean_triggers: return np.nan
    ttrs = []
    y_vals, m_vals, s_vals = y_series.values, m_series.values, s_scores.values
    max_len = len(y_vals)
    
    for t in clean_triggers:
        anchor_m = m_vals[t]
        direction = -1 if s_vals[t] > 0 else 1
        max_search = min(t + 2880, max_len)
        forward_y = y_vals[t+1 : max_search]
        
        if direction == -1: crossings = np.where(forward_y < anchor_m)[0]
        else: crossings = np.where(forward_y > anchor_m)[0]
            
        if len(crossings) > 0: ttrs.append(crossings[0] + 1)
            
    return np.median(ttrs) if ttrs else np.nan

def run_part4_theoretical_vs_empirical(df, all_pairs):
    print("\n" + "="*125)
    print(" 📘 PART 4: MODEL USEFULNESS & RANK CORRELATION (THEORETICAL VS EMPIRICAL TTR)")
    print("="*125)
    print(" 📖 METRICS GLOSSARY:")
    print(" - Theo Half-Life : Mathematical half-life of the AR(1) decay.")
    print(" - Anchored TTR   : Realized physical time-to-reversion once a sigma threshold is breached.")
    print(" - Spearman Corr  : Proves if pairs with mathematically fast decay actually decay faster in physical reality.")
    print("="*125)

    results = []
    for exch_A, exch_B in all_pairs:
        pair_col = f"{ADJ_SPREAD_PREFIX}_{exch_A}_vs_{exch_B}"
        if pair_col not in df.columns: continue
            
        stale_A = df.get(f'stale_{exch_A}', pd.Series(True, index=df.index))
        stale_B = df.get(f'stale_{exch_B}', pd.Series(True, index=df.index))
        active_mask = (~stale_A) & (~stale_B)
        pair_name = f"{exch_A.split('-')[0].upper()[:4]}/{exch_B.split('-')[0].upper()[:4]}"
        
        y = df[pair_col].where(active_mask)
        x = y.shift(1)
        
        roll_mean = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).mean()
        roll_cov = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).cov(x)
        roll_var_x = x.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).var()
        roll_var_y = y.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).var()
        
        rho_1_series = (roll_cov / roll_var_x).clip(lower=1e-5, upper=0.999)
        avg_rho_1 = rho_1_series.mean()
        kappa = -np.log(avg_rho_1) if avg_rho_1 > 0 and pd.notna(avg_rho_1) else np.nan
        theo_hl = np.log(2) / kappa if pd.notna(kappa) and kappa > 0 else np.nan
        
        m = (roll_mean - rho_1_series * x.rolling(ROLLING_WINDOW_MINS, min_periods=MIN_VALID_MINS).mean()) / (1 - rho_1_series)
        var_e = (roll_var_y - (rho_1_series**2) * roll_var_x).clip(lower=1e-8)
        sigma_eq = np.sqrt(var_e / (1 - rho_1_series**2))
        s_scores = ((y - m) / sigma_eq)
        
        if len(s_scores.dropna()) < 10000: continue
        
        ttr_1s = calculate_anchored_empirical_ttr(y, m, s_scores, 1.0)
        ttr_2s = calculate_anchored_empirical_ttr(y, m, s_scores, 2.0)
        ttr_3s = calculate_anchored_empirical_ttr(y, m, s_scores, 3.0)
        
        results.append({
            'Pair': pair_name, 'Theo Half-Life': theo_hl,
            'TTR 1σ': ttr_1s, 'TTR 2σ': ttr_2s, 'TTR 3σ': ttr_3s
        })

    if not results: return print("[!] No pairs met the minimum sample size constraint.")

    res_df = pd.DataFrame(results).sort_values('Theo Half-Life')
    corr_1s = spearmanr(res_df['Theo Half-Life'], res_df['TTR 1σ'], nan_policy='omit')[0]
    corr_2s = spearmanr(res_df['Theo Half-Life'], res_df['TTR 2σ'], nan_policy='omit')[0]
    corr_3s = spearmanr(res_df['Theo Half-Life'], res_df['TTR 3σ'], nan_policy='omit')[0]
    
    fmt = lambda x: f"{x:.1f}" if pd.notna(x) else "N/A"
    for c in ['Theo Half-Life', 'TTR 1σ', 'TTR 2σ', 'TTR 3σ']: res_df[c] = res_df[c].apply(fmt)
            
    print(res_df.to_string(index=False))
    print("="*125)
    print(f" 📈 SPEARMAN RANK CORRELATION (Theo HL vs 1σ Bulk Regime TTR): {corr_1s:.3f}")
    print(f" 📈 SPEARMAN RANK CORRELATION (Theo HL vs 2σ Void Regime TTR): {corr_2s:.3f}")
    print(f" 📈 SPEARMAN RANK CORRELATION (Theo HL vs 3σ Tail Regime TTR): {corr_3s:.3f}")
    print("="*125)

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
if __name__ == "__main__":
    print("🚀 Ingesting Master Dataset...")
    df = pd.read_parquet(MASTER_PARQUET).loc[GLOBAL_START:GLOBAL_END]
    exchanges = [c.split('_')[-1] for c in df.columns if c.startswith('mid_')]
    all_pairs = list(itertools.combinations(exchanges, 2))
    
    run_part1_stylized_facts(df, all_pairs)
    run_part2_signal_frequency(df, all_pairs)
    run_part3_ou_validation(df, all_pairs)
    run_part4_theoretical_vs_empirical(df, all_pairs)

Data Directory: C:\Users\tz_ho\crypto-perpetual-arbitrage\data
Master Parquet: C:\Users\tz_ho\crypto-perpetual-arbitrage\data\BTC_TAGGED_MASTER.parquet
🚀 Ingesting Master Dataset...

 📘 PART 1: MACRO STYLIZED FACTS & MINIMUM ARBITRAGE BANDS (MAB)
 ⚙️  APPROACH: STATIC EX-POST WITH DAILY ROLLING SAMPLING
 📖 METRICS GLOSSARY:
 - StdDev(bps)    : The unconditional volatility of the spread.
 - MAB(bps)       : Minimum Arbitrage Band. (Round-Trip Taker Fees + Assumed 2.0 bps Slippage).
 - Req. Sigma     : (MAB / StdDev). The theoretical Z-score magnitude required just to break even.
 - Static ADF     : Full-year Dickey-Fuller test (proves global cointegration).
 - Roll Stat %    : Percentage of rolling 24h windows that passed the ADF test (proves continuous tradability).
     Pair   Mean StdDev MAB(bps) Req. Sigma ACF(1) Static ADF Roll Stat % Active Days
BYBI/GATE  0.500  2.047    10.00       4.89 0.9682     0.0000       96.4%       137.4
BINA/BYBI -1.167  2.074    10.00       4.82 0.9547 

In [ ]:
# ==============================================================================
# NAIVE Z-SCORE VS NORMALIZED OU S-SCORE COMPARISON (BASELINE 0.2 MECHANICS)
# ==============================================================================
import pandas as pd
import numpy as np
import itertools
from pathlib import Path
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. PARAMETRIC PORTFOLIO CONFIGURATION
# ==============================================================================
DATA_DIR = Path(r"C:\Users\tz_ho\research_data")
TOKEN = 'BTC'

EMP_START = pd.Timestamp('2025-03-01 00:00:00') 
EMP_END = pd.Timestamp('2025-12-31 23:59:00')
ALL_CALENDAR_DAYS = pd.date_range(start=EMP_START.date(), end=EMP_END.date(), freq='D').date
DAYS_TOTAL = (EMP_END - EMP_START).days

ROLLING_WINDOW_MINS = 1440  
TRADE_TIMEOUT_MINS = 300
SIGMAS_TO_TEST = [3.0, 3.5, 4.0] 

FEES = {
    'binf': 1.70, 'okex': 1.75, 'byde': 3.20, 
    'kusw': 2.50, 'gate': 2.00, 'hype': 1.44
}

NAME_MAP = {
    'binance-futures': 'binf', 'bybit': 'byde', 'okex-swap': 'okex', 
    'gate-io-futures': 'gate', 'hyperliquid': 'hype', 'kucoin-futures': 'kusw'
}

# ==============================================================================
# 2. NATIVE DATA UTILITIES 
# ==============================================================================
def robust_to_datetime(series):
    s_num = pd.to_numeric(series, errors='coerce')
    med_val = s_num.median()
    if pd.isna(med_val): return pd.to_datetime(series, errors='coerce', utc=True).dt.tz_localize(None)
    unit = 'ns' if med_val > 1e16 else ('us' if med_val > 1e13 else ('ms' if med_val > 1e10 else 's'))
    return pd.to_datetime(s_num, unit=unit, errors='coerce', utc=True).dt.floor('1min').dt.tz_localize(None)

def load_aligned_bbo(df_master, exch_A, exch_B, bbo_paths):
    df_pair = pd.DataFrame(index=df_master.index)
    for exch in [exch_A, exch_B]:
        bbo = pd.read_csv(bbo_paths[exch], usecols=['minute', 'bid', 'ask'])
        bbo['timestamp'] = robust_to_datetime(bbo['minute'])
        bbo.set_index('timestamp', inplace=True)
        bbo = bbo[['bid', 'ask']].copy()
        
        bbo.loc[bbo['bid'] >= bbo['ask'], ['bid', 'ask']] = np.nan
        bbo = bbo.reindex(df_master.index).ffill(limit=3)
        df_pair[f'bid_{exch}'] = bbo['bid']
        df_pair[f'ask_{exch}'] = bbo['ask']
        df_pair[f'spread_{exch}'] = (np.log(bbo['ask']) - np.log(bbo['bid'])) * 10000

    df_pair['combined_spread'] = df_pair[f'spread_{exch_A}'] + df_pair[f'spread_{exch_B}']
    df_pair['liquidity_toll'] = df_pair['combined_spread'].rolling(1440).median().bfill()
    return df_pair

def calc_stochastic_ex_ante(df_master, adj_col, raw_col, w):
    df = pd.DataFrame(index=df_master.index)
    df[adj_col] = df_master[adj_col]
    df['raw_ma'] = df_master[raw_col].shift(1).rolling(w).mean()
    
    y_prev, x_prev = df[adj_col].shift(1), df[adj_col].shift(2)
    roll_cov = y_prev.rolling(w).cov(x_prev)
    roll_var_x = x_prev.rolling(w).var()
    roll_var_y = y_prev.rolling(w).var()
    mean_x, mean_y = x_prev.rolling(w).mean(), y_prev.rolling(w).mean()
    
    phi = (roll_cov / roll_var_x).clip(upper=0.999, lower=-0.999) 
    a = mean_y - phi * mean_x
    m = a / (1 - phi)
    var_e = (roll_var_y - (phi**2) * roll_var_x).clip(lower=1e-8)
    sigma_eq = np.sqrt(var_e / (1 - phi**2))
    
    df['s_score'] = (df[adj_col] - m) / sigma_eq
    
    # NAIVE Z-SCORE CALCULATION
    x_raw_prev = df_master[raw_col].shift(1)
    raw_roll_mean = x_raw_prev.rolling(w).mean()
    raw_roll_std = x_raw_prev.rolling(w).std()
    df['naive_z'] = (df_master[raw_col] - raw_roll_mean) / raw_roll_std
    
    return df['s_score'].values, df['naive_z'].values, df['raw_ma'].values

# ==============================================================================
# 3. CHRONOLOGICAL ENGINE LAYER
# ==============================================================================
def run_engine_for_sigma_and_type(sigma_threshold, signal_type, df_master, signals, bbo_storage, timestamps):
    n_steps = len(df_master)
    active_positions, trade_log = {}, []
    last_entry_t = {pair: -60 for pair in signals.keys()}

    for t in range(ROLLING_WINDOW_MINS, n_steps):
        current_time = timestamps[t]
        
        # ==========================================
        # 1. EVALUATE POSITION EXITS
        # ==========================================
        keys_to_remove = []
        for pair, pos in active_positions.items():
            sig_data = signals[pair]
            bbo_df = bbo_storage[pair]
            exch_a, exch_b = pair.split('/')
            
            hold_time = t - pos['entry_idx']
            timeout_hit = hold_time >= TRADE_TIMEOUT_MINS
            funding_hit = sig_data['is_funding'][t] 
            ma_reverted = (sig_data['raw_mid'][t] <= pos['target_ma']) if pos['direction'] == -1 else (sig_data['raw_mid'][t] >= pos['target_ma'])
            
            bid_a_val, ask_a_val = bbo_df[f'bid_{exch_a.lower()}'].iloc[t], bbo_df[f'ask_{exch_a.lower()}'].iloc[t]
            bid_b_val, ask_b_val = bbo_df[f'bid_{exch_b.lower()}'].iloc[t], bbo_df[f'ask_{exch_b.lower()}'].iloc[t]
            is_bbo_invalid = np.isnan(bid_a_val) or np.isnan(ask_a_val) or np.isnan(bid_b_val) or np.isnan(ask_b_val)
            
            must_close = False
            exit_reason = None
            
            if funding_hit: must_close, exit_reason = True, "Funding_Epoch"
            elif timeout_hit: must_close, exit_reason = True, "Timeout"
            elif ma_reverted: must_close, exit_reason = True, "MA_Hit"
            elif pos.get('force_close_pending', False): must_close, exit_reason = True, pos.get('pending_reason')
            
            if must_close:
                if is_bbo_invalid:
                    if hold_time >= TRADE_TIMEOUT_MINS or exit_reason in ["Funding_Epoch", "Funding_Delayed"]:
                        valid_bbo = bbo_df.iloc[:t].dropna(subset=[
                            f'bid_{exch_a.lower()}', f'ask_{exch_a.lower()}', 
                            f'bid_{exch_b.lower()}', f'ask_{exch_b.lower()}'
                        ])
                        if not valid_bbo.empty:
                            last_valid = valid_bbo.iloc[-1]
                            bid_a_val, ask_a_val = last_valid[f'bid_{exch_a.lower()}'], last_valid[f'ask_{exch_a.lower()}']
                            bid_b_val, ask_b_val = last_valid[f'bid_{exch_b.lower()}'], last_valid[f'ask_{exch_b.lower()}']
                            
                            if exit_reason in ["Funding_Epoch", "Funding_Delayed"]:
                                pos['pending_reason'] = "Funding_Imputed"
                            else:
                                pos['pending_reason'] = "Timeout_Imputed"
                        else:
                            bid_a_val, ask_a_val, bid_b_val, ask_b_val = 1.0, 1.0, 1.0, 1.0
                    else:
                        pos['force_close_pending'] = True
                        if 'pending_reason' not in pos:
                            pos['pending_reason'] = "MA_Hit_Delayed"
                        continue 
                
                if pos.get('force_close_pending', False) or (is_bbo_invalid and (hold_time >= TRADE_TIMEOUT_MINS or exit_reason in ["Funding_Epoch", "Funding_Delayed"])):
                    exit_reason = pos.get('pending_reason', "Timeout_Imputed")
                elif funding_hit: exit_reason = "Funding_Epoch"
                elif timeout_hit: exit_reason = "Timeout"
                else: exit_reason = "MA_Hit"
                
                if pos['direction'] == -1:  
                    current_z_exec = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
                    gross_pnl = pos['z_entry_exec'] - current_z_exec
                else:                      
                    current_z_exec = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
                    gross_pnl = current_z_exec - pos['z_entry_exec']
                
                net_pnl = gross_pnl - pos['round_trip_fee_bps']
                
                trade_log.append({
                    'Pair': pair, 'Entry_Time': timestamps[pos['entry_idx']], 'Exit_Time': current_time, 
                    'Hold_Time': hold_time, 'Gross_PnL': gross_pnl, 'Net_PnL': net_pnl, 'Reason': exit_reason
                })
                keys_to_remove.append(pair)
                
        for k in keys_to_remove: del active_positions[k]

        # ==========================================
        # 2. EVALUATE POSITION ENTRIES
        # ==========================================
        for pair, sig_data in signals.items():
            if pair in active_positions or t - last_entry_t[pair] < 60 or sig_data['is_funding'][t]: continue  
                
            obs_s = sig_data['s_score'][t] if signal_type == 'ou' else sig_data['naive_z'][t]
            
            if np.isnan(obs_s) or np.abs(obs_s) < sigma_threshold: continue
                
            bbo_df = bbo_storage[pair]
            exch_a, exch_b = pair.split('/')
            
            bid_a_val, ask_a_val = bbo_df[f'bid_{exch_a.lower()}'].iloc[t], bbo_df[f'ask_{exch_a.lower()}'].iloc[t]
            bid_b_val, ask_b_val = bbo_df[f'bid_{exch_b.lower()}'].iloc[t], bbo_df[f'ask_{exch_b.lower()}'].iloc[t]
            if np.isnan(bid_a_val) or np.isnan(ask_a_val) or np.isnan(bid_b_val) or np.isnan(ask_b_val): continue
                
            direction = -1 if obs_s > 0 else 1
            target_ma = sig_data['raw_ma'][t]
            
            if direction == -1:
                entry_phys = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
                gross_ev = entry_phys - target_ma
            else:
                entry_phys = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
                gross_ev = target_ma - entry_phys
                
            toll = sig_data['liquidity_toll'][t]
            net_ev = gross_ev - sig_data['round_trip_fee_bps'] - toll
            
            if net_ev <= 0: continue  
                
            active_positions[pair] = {
                'entry_idx': t, 'z_entry_exec': entry_phys, 
                'target_ma': target_ma, 'direction': direction, 
                'round_trip_fee_bps': sig_data['round_trip_fee_bps']
            }
            last_entry_t[pair] = t  

    # ==============================================================================
    # 🟢 TERMINAL LIQUIDATION BLOCK (SOLVES 912m NaN ZOMBIE HOLD BUG)
    # ==============================================================================
    terminal_idx = n_steps - 1
    terminal_time = timestamps[terminal_idx]
    
    for pair, pos in list(active_positions.items()):
        bbo_df = bbo_storage[pair]
        exch_a, exch_b = pair.split('/')
        hold_time = min(terminal_idx - pos['entry_idx'], TRADE_TIMEOUT_MINS)
        
        valid_bbo = bbo_df.dropna(subset=[f'bid_{exch_a.lower()}', f'ask_{exch_a.lower()}', f'bid_{exch_b.lower()}', f'ask_{exch_b.lower()}'])
        if not valid_bbo.empty:
            last_valid_row = valid_bbo.iloc[-1]
            bid_a_val, ask_a_val = last_valid_row[f'bid_{exch_a.lower()}'], last_valid_row[f'ask_{exch_a.lower()}']
            bid_b_val, ask_b_val = last_valid_row[f'bid_{exch_b.lower()}'], last_valid_row[f'ask_{exch_b.lower()}']
        else:
            bid_a_val, ask_a_val, bid_b_val, ask_b_val = 1.0, 1.0, 1.0, 1.0
            
        if pos['direction'] == -1:
            current_z_exec = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
            gross_pnl = pos['z_entry_exec'] - current_z_exec
        else:
            current_z_exec = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
            gross_pnl = current_z_exec - pos['z_entry_exec']
            
        net_pnl = gross_pnl - pos['round_trip_fee_bps']
        
        trade_log.append({
            'Pair': pair, 'Entry_Time': timestamps[pos['entry_idx']], 'Exit_Time': terminal_time,
            'Hold_Time': hold_time, 'Gross_PnL': gross_pnl, 'Net_PnL': net_pnl, 'Reason': "Terminal_Force_Clean"
        })

    return pd.DataFrame(trade_log)

# ==============================================================================
# 4. ORCHESTRATOR
# ==============================================================================
def run_comparison():
    print(f"🚀 Loading Tagged Master Parquet for {TOKEN}...")
    master_parquet = DATA_DIR / f"{TOKEN}_TAGGED_MASTER.parquet"
    if not master_parquet.exists():
        print(f"[!] Warning: Master Parquet for {TOKEN} not found.")
        return
        
    df_master = pd.read_parquet(master_parquet).loc[EMP_START:EMP_END]
    
    bbo_paths = {
        'binf': DATA_DIR / rf"binf_{TOKEN}_bbo.csv",
        'byde': DATA_DIR / rf"byde_{TOKEN}_bbo.csv",
        'okex': DATA_DIR / rf"okex_{TOKEN}_bbo.csv",
        'gate': DATA_DIR / rf"gate_{TOKEN}_bbo.csv",
        'hype': DATA_DIR / rf"hype_{TOKEN}_bbo.csv",
        'kusw': DATA_DIR / rf"kusw_{TOKEN}_bbo.csv"
    }
    
    exchanges = list(bbo_paths.keys())
    pairs = list(itertools.combinations(exchanges, 2))
    
    signals, bbo_storage = {}, {}
    n_steps = len(df_master)
    timestamps = df_master.index
    
    def get_data_driven_epoch(exch_name, full_name):
        epoch_col = f'epoch_id_{full_name}'
        if epoch_col not in df_master.columns:
            epoch_col = f'epoch_id_{exch_name}'
        if epoch_col in df_master.columns:
            shift_mask = df_master[epoch_col] != df_master[epoch_col].shift(-1)
            shift_mask.iloc[-1] = False
            return shift_mask.values, shift_mask.sum()
        return np.zeros(n_steps, dtype=bool), 0

    for exch_a, exch_b in tqdm(pairs, desc="Pre-computing Dual Footprints"):
        pair_label = f"{exch_a.upper()}/{exch_b.upper()}"
        full_name_A = [k for k, v in NAME_MAP.items() if v == exch_a][0]
        full_name_B = [k for k, v in NAME_MAP.items() if v == exch_b][0]
        
        adj_col = f'spread_padj_idx_mid_avg_{full_name_A}_vs_{full_name_B}'
        raw_col = f'spread_raw_mid_{full_name_A}_vs_{full_name_B}'
        
        if adj_col not in df_master.columns or raw_col not in df_master.columns: continue
            
        bbo_df = load_aligned_bbo(df_master, exch_a, exch_b, bbo_paths)
        s_score, naive_z, raw_ma = calc_stochastic_ex_ante(df_master, adj_col, raw_col, ROLLING_WINDOW_MINS)
        z_exec_raw = df_master[raw_col].values
        
        mask_a = (timestamps.minute == 59) if exch_a == 'hype' else get_data_driven_epoch(exch_a, full_name_A)[0]
        mask_b = (timestamps.minute == 59) if exch_b == 'hype' else get_data_driven_epoch(exch_b, full_name_B)[0]
        is_funding_epoch = mask_a | mask_b

        signals[pair_label] = {
            's_score': s_score, 'naive_z': naive_z, 'raw_ma': raw_ma, 'raw_mid': z_exec_raw,
            'is_funding': is_funding_epoch,
            'liquidity_toll': bbo_df['liquidity_toll'].values,
            'round_trip_fee_bps': (FEES[exch_a] + FEES[exch_b]) * 2
        }
        bbo_storage[pair_label] = bbo_df

    sweep_results = []

    print("\n🔬 Commencing Head-to-Head Sensitivity Sweep...")
    
    for sig_type in ['naive', 'ou']:
        model_name = "Naive Raw Z-Score" if sig_type == 'naive' else "Normalized OU S-Score"
        for sig_thresh in tqdm(SIGMAS_TO_TEST, desc=f"Testing {model_name}"):
            tdf = run_engine_for_sigma_and_type(sig_thresh, sig_type, df_master, signals, bbo_storage, timestamps)
            
            if tdf.empty:
                sweep_results.append({'Model': model_name, 'Sigma': f"{sig_thresh}σ", 'Trades': 0, 'Win%': 0.0, 'Cum_PnL(bps)': 0.0, 'CalSR': 0.0, 'ActSR': 0.0, 'TrdSR': 0.0, 'MDD(bps)': 0.0, 'G2P': 0.0})
                continue
                
            tdf['close_date'] = pd.to_datetime(tdf['Exit_Time']).dt.date
            active_daily_bps = tdf.groupby('close_date')['Net_PnL'].sum()
            calendar_daily_bps = active_daily_bps.reindex(ALL_CALENDAR_DAYS, fill_value=0.0)
            
            global_cum_bps = calendar_daily_bps.sum()
            global_peaks = calendar_daily_bps.cumsum().cummax()
            global_mdd_bps = (global_peaks - calendar_daily_bps.cumsum()).max()
            win_pct = (tdf['Net_PnL'] > 0).mean() * 100
            
            cal_sharpe = (calendar_daily_bps.mean() / max(calendar_daily_bps.std(), 1e-6)) * np.sqrt(365)
            act_sharpe = (active_daily_bps.mean() / max(active_daily_bps.std(), 1e-6)) * np.sqrt(365) if len(active_daily_bps) > 1 else 0.0
            trd_sharpe = tdf['Net_PnL'].mean() / max(tdf['Net_PnL'].std(), 1e-6)
            
            gross_gains = tdf.loc[tdf['Net_PnL'] > 0, 'Net_PnL'].sum()
            gross_losses = abs(tdf.loc[tdf['Net_PnL'] < 0, 'Net_PnL'].sum())
            g2p_ratio = (gross_gains / gross_losses) if gross_losses > 0 else 999.99
            
            sweep_results.append({
                'Model': model_name,
                'Sigma': f"{sig_thresh}σ",
                'Trades': len(tdf),
                'Win%': round(win_pct, 2),
                'Cum_PnL(bps)': round(global_cum_bps, 2),
                'CalSR': round(cal_sharpe, 2),
                'ActSR': round(act_sharpe, 2),
                'TrdSR': round(trd_sharpe, 2),
                'MDD(bps)': round(global_mdd_bps, 2),
                'G2P': round(g2p_ratio, 2)
            })

    print("\n" + "="*125)
    print(" 📊 HEAD-TO-HEAD COMPARISON: NAIVE VS NORMALIZED ENGINE (UNCONSTRAINED BTC)")
    print("="*125)
    summary_df = pd.DataFrame(sweep_results)
    print(summary_df.to_string(index=False))
    print("="*125 + "\n")

if __name__ == "__main__":
    run_comparison()

In [7]:
# ==============================================================================
# NAIVE Z-SCORE VS NORMALIZED OU S-SCORE COMPARISON (BASELINE 0.2 MECHANICS)
# ==============================================================================
import pandas as pd
import numpy as np
import itertools
from pathlib import Path
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. PARAMETRIC PORTFOLIO CONFIGURATION & DYNAMIC PATHS
# ==============================================================================
# Dynamically locate the data folder relative to the current notebook
NOTEBOOK_DIR = Path.cwd()               # e.g., .../crypto-perpetual-arbitrage/notebooks
PROJECT_DIR = NOTEBOOK_DIR.parent       # e.g., .../crypto-perpetual-arbitrage
DATA_DIR = PROJECT_DIR / "data"         # e.g., .../crypto-perpetual-arbitrage/data

TOKEN = 'BTC'

EMP_START = pd.Timestamp('2025-03-01 00:00:00') 
EMP_END = pd.Timestamp('2025-12-31 23:59:00')
ALL_CALENDAR_DAYS = pd.date_range(start=EMP_START.date(), end=EMP_END.date(), freq='D').date
DAYS_TOTAL = (EMP_END - EMP_START).days

ROLLING_WINDOW_MINS = 1440  
TRADE_TIMEOUT_MINS = 300
SIGMAS_TO_TEST = [3.0, 3.5, 4.0] 

FEES = {
    'binf': 1.70, 'okex': 1.75, 'byde': 3.20, 
    'kusw': 2.50, 'gate': 2.00, 'hype': 1.44
}

NAME_MAP = {
    'binance-futures': 'binf', 'bybit': 'byde', 'okex-swap': 'okex', 
    'gate-io-futures': 'gate', 'hyperliquid': 'hype', 'kucoin-futures': 'kusw'
}

# ==============================================================================
# 2. NATIVE DATA UTILITIES 
# ==============================================================================
def robust_to_datetime(series):
    s_num = pd.to_numeric(series, errors='coerce')
    med_val = s_num.median()
    if pd.isna(med_val): return pd.to_datetime(series, errors='coerce', utc=True).dt.tz_localize(None)
    unit = 'ns' if med_val > 1e16 else ('us' if med_val > 1e13 else ('ms' if med_val > 1e10 else 's'))
    return pd.to_datetime(s_num, unit=unit, errors='coerce', utc=True).dt.floor('1min').dt.tz_localize(None)

def load_aligned_bbo(df_master, exch_A, exch_B, bbo_paths):
    df_pair = pd.DataFrame(index=df_master.index)
    for exch in [exch_A, exch_B]:
        bbo = pd.read_csv(bbo_paths[exch], usecols=['minute', 'bid', 'ask'])
        bbo['timestamp'] = robust_to_datetime(bbo['minute'])
        bbo.set_index('timestamp', inplace=True)
        bbo = bbo[['bid', 'ask']].copy()
        
        bbo.loc[bbo['bid'] >= bbo['ask'], ['bid', 'ask']] = np.nan
        bbo = bbo.reindex(df_master.index).ffill(limit=3)
        df_pair[f'bid_{exch}'] = bbo['bid']
        df_pair[f'ask_{exch}'] = bbo['ask']
        df_pair[f'spread_{exch}'] = (np.log(bbo['ask']) - np.log(bbo['bid'])) * 10000

    df_pair['combined_spread'] = df_pair[f'spread_{exch_A}'] + df_pair[f'spread_{exch_B}']
    df_pair['liquidity_toll'] = df_pair['combined_spread'].rolling(1440).median().bfill()
    return df_pair

def calc_stochastic_ex_ante(df_master, adj_col, raw_col, w):
    df = pd.DataFrame(index=df_master.index)
    df[adj_col] = df_master[adj_col]
    df['raw_ma'] = df_master[raw_col].shift(1).rolling(w).mean()
    
    y_prev, x_prev = df[adj_col].shift(1), df[adj_col].shift(2)
    roll_cov = y_prev.rolling(w).cov(x_prev)
    roll_var_x = x_prev.rolling(w).var()
    roll_var_y = y_prev.rolling(w).var()
    mean_x, mean_y = x_prev.rolling(w).mean(), y_prev.rolling(w).mean()
    
    phi = (roll_cov / roll_var_x).clip(upper=0.999, lower=-0.999) 
    a = mean_y - phi * mean_x
    m = a / (1 - phi)
    var_e = (roll_var_y - (phi**2) * roll_var_x).clip(lower=1e-8)
    sigma_eq = np.sqrt(var_e / (1 - phi**2))
    
    df['s_score'] = (df[adj_col] - m) / sigma_eq
    
    # NAIVE Z-SCORE CALCULATION
    x_raw_prev = df_master[raw_col].shift(1)
    raw_roll_mean = x_raw_prev.rolling(w).mean()
    raw_roll_std = x_raw_prev.rolling(w).std()
    df['naive_z'] = (df_master[raw_col] - raw_roll_mean) / raw_roll_std
    
    return df['s_score'].values, df['naive_z'].values, df['raw_ma'].values

# ==============================================================================
# 3. CHRONOLOGICAL ENGINE LAYER
# ==============================================================================
def run_engine_for_sigma_and_type(sigma_threshold, signal_type, df_master, signals, bbo_storage, timestamps):
    n_steps = len(df_master)
    active_positions, trade_log = {}, []
    last_entry_t = {pair: -60 for pair in signals.keys()}

    for t in range(ROLLING_WINDOW_MINS, n_steps):
        current_time = timestamps[t]
        
        # ==========================================
        # 1. EVALUATE POSITION EXITS
        # ==========================================
        keys_to_remove = []
        for pair, pos in active_positions.items():
            sig_data = signals[pair]
            bbo_df = bbo_storage[pair]
            exch_a, exch_b = pair.split('/')
            
            hold_time = t - pos['entry_idx']
            timeout_hit = hold_time >= TRADE_TIMEOUT_MINS
            funding_hit = sig_data['is_funding'][t] 
            ma_reverted = (sig_data['raw_mid'][t] <= pos['target_ma']) if pos['direction'] == -1 else (sig_data['raw_mid'][t] >= pos['target_ma'])
            
            bid_a_val, ask_a_val = bbo_df[f'bid_{exch_a.lower()}'].iloc[t], bbo_df[f'ask_{exch_a.lower()}'].iloc[t]
            bid_b_val, ask_b_val = bbo_df[f'bid_{exch_b.lower()}'].iloc[t], bbo_df[f'ask_{exch_b.lower()}'].iloc[t]
            is_bbo_invalid = np.isnan(bid_a_val) or np.isnan(ask_a_val) or np.isnan(bid_b_val) or np.isnan(ask_b_val)
            
            must_close = False
            exit_reason = None
            
            if funding_hit: must_close, exit_reason = True, "Funding_Epoch"
            elif timeout_hit: must_close, exit_reason = True, "Timeout"
            elif ma_reverted: must_close, exit_reason = True, "MA_Hit"
            elif pos.get('force_close_pending', False): must_close, exit_reason = True, pos.get('pending_reason')
            
            if must_close:
                if is_bbo_invalid:
                    if hold_time >= TRADE_TIMEOUT_MINS or exit_reason in ["Funding_Epoch", "Funding_Delayed"]:
                        valid_bbo = bbo_df.iloc[:t].dropna(subset=[
                            f'bid_{exch_a.lower()}', f'ask_{exch_a.lower()}', 
                            f'bid_{exch_b.lower()}', f'ask_{exch_b.lower()}'
                        ])
                        if not valid_bbo.empty:
                            last_valid = valid_bbo.iloc[-1]
                            bid_a_val, ask_a_val = last_valid[f'bid_{exch_a.lower()}'], last_valid[f'ask_{exch_a.lower()}']
                            bid_b_val, ask_b_val = last_valid[f'bid_{exch_b.lower()}'], last_valid[f'ask_{exch_b.lower()}']
                            
                            if exit_reason in ["Funding_Epoch", "Funding_Delayed"]:
                                pos['pending_reason'] = "Funding_Imputed"
                            else:
                                pos['pending_reason'] = "Timeout_Imputed"
                        else:
                            bid_a_val, ask_a_val, bid_b_val, ask_b_val = 1.0, 1.0, 1.0, 1.0
                    else:
                        pos['force_close_pending'] = True
                        if 'pending_reason' not in pos:
                            pos['pending_reason'] = "MA_Hit_Delayed"
                        continue 
                
                if pos.get('force_close_pending', False) or (is_bbo_invalid and (hold_time >= TRADE_TIMEOUT_MINS or exit_reason in ["Funding_Epoch", "Funding_Delayed"])):
                    exit_reason = pos.get('pending_reason', "Timeout_Imputed")
                elif funding_hit: exit_reason = "Funding_Epoch"
                elif timeout_hit: exit_reason = "Timeout"
                else: exit_reason = "MA_Hit"
                
                if pos['direction'] == -1:  
                    current_z_exec = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
                    gross_pnl = pos['z_entry_exec'] - current_z_exec
                else:                      
                    current_z_exec = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
                    gross_pnl = current_z_exec - pos['z_entry_exec']
                
                net_pnl = gross_pnl - pos['round_trip_fee_bps']
                
                trade_log.append({
                    'Pair': pair, 'Entry_Time': timestamps[pos['entry_idx']], 'Exit_Time': current_time, 
                    'Hold_Time': hold_time, 'Gross_PnL': gross_pnl, 'Net_PnL': net_pnl, 'Reason': exit_reason
                })
                keys_to_remove.append(pair)
                
        for k in keys_to_remove: del active_positions[k]

        # ==========================================
        # 2. EVALUATE POSITION ENTRIES
        # ==========================================
        for pair, sig_data in signals.items():
            if pair in active_positions or t - last_entry_t[pair] < 60 or sig_data['is_funding'][t]: continue  
                
            obs_s = sig_data['s_score'][t] if signal_type == 'ou' else sig_data['naive_z'][t]
            
            if np.isnan(obs_s) or np.abs(obs_s) < sigma_threshold: continue
                
            bbo_df = bbo_storage[pair]
            exch_a, exch_b = pair.split('/')
            
            bid_a_val, ask_a_val = bbo_df[f'bid_{exch_a.lower()}'].iloc[t], bbo_df[f'ask_{exch_a.lower()}'].iloc[t]
            bid_b_val, ask_b_val = bbo_df[f'bid_{exch_b.lower()}'].iloc[t], bbo_df[f'ask_{exch_b.lower()}'].iloc[t]
            if np.isnan(bid_a_val) or np.isnan(ask_a_val) or np.isnan(bid_b_val) or np.isnan(ask_b_val): continue
                
            direction = -1 if obs_s > 0 else 1
            target_ma = sig_data['raw_ma'][t]
            
            if direction == -1:
                entry_phys = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
                gross_ev = entry_phys - target_ma
            else:
                entry_phys = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
                gross_ev = target_ma - entry_phys
                
            toll = sig_data['liquidity_toll'][t]
            net_ev = gross_ev - sig_data['round_trip_fee_bps'] - toll
            
            if net_ev <= 0: continue  
                
            active_positions[pair] = {
                'entry_idx': t, 'z_entry_exec': entry_phys, 
                'target_ma': target_ma, 'direction': direction, 
                'round_trip_fee_bps': sig_data['round_trip_fee_bps']
            }
            last_entry_t[pair] = t  

    # ==============================================================================
    # 🟢 TERMINAL LIQUIDATION BLOCK (SOLVES 912m NaN ZOMBIE HOLD BUG)
    # ==============================================================================
    terminal_idx = n_steps - 1
    terminal_time = timestamps[terminal_idx]
    
    for pair, pos in list(active_positions.items()):
        bbo_df = bbo_storage[pair]
        exch_a, exch_b = pair.split('/')
        hold_time = min(terminal_idx - pos['entry_idx'], TRADE_TIMEOUT_MINS)
        
        valid_bbo = bbo_df.dropna(subset=[f'bid_{exch_a.lower()}', f'ask_{exch_a.lower()}', f'bid_{exch_b.lower()}', f'ask_{exch_b.lower()}'])
        if not valid_bbo.empty:
            last_valid_row = valid_bbo.iloc[-1]
            bid_a_val, ask_a_val = last_valid_row[f'bid_{exch_a.lower()}'], last_valid_row[f'ask_{exch_a.lower()}']
            bid_b_val, ask_b_val = last_valid_row[f'bid_{exch_b.lower()}'], last_valid_row[f'ask_{exch_b.lower()}']
        else:
            bid_a_val, ask_a_val, bid_b_val, ask_b_val = 1.0, 1.0, 1.0, 1.0
            
        if pos['direction'] == -1:
            current_z_exec = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
            gross_pnl = pos['z_entry_exec'] - current_z_exec
        else:
            current_z_exec = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
            gross_pnl = current_z_exec - pos['z_entry_exec']
            
        net_pnl = gross_pnl - pos['round_trip_fee_bps']
        
        trade_log.append({
            'Pair': pair, 'Entry_Time': timestamps[pos['entry_idx']], 'Exit_Time': terminal_time,
            'Hold_Time': hold_time, 'Gross_PnL': gross_pnl, 'Net_PnL': net_pnl, 'Reason': "Terminal_Force_Clean"
        })

    return pd.DataFrame(trade_log)

# ==============================================================================
# 4. ORCHESTRATOR
# ==============================================================================
def run_comparison():
    print(f"🚀 Loading Tagged Master Parquet for {TOKEN} from:\n   {DATA_DIR}")
    master_parquet = DATA_DIR / f"{TOKEN}_TAGGED_MASTER.parquet"
    if not master_parquet.exists():
        print(f"[!] Warning: Master Parquet for {TOKEN} not found.")
        return
        
    df_master = pd.read_parquet(master_parquet).loc[EMP_START:EMP_END]
    
    # Fully dynamic, cross-platform paths using pathlib
    bbo_paths = {
        'binf': DATA_DIR / f"binf_{TOKEN}_bbo.csv",
        'byde': DATA_DIR / f"byde_{TOKEN}_bbo.csv",
        'okex': DATA_DIR / f"okex_{TOKEN}_bbo.csv",
        'gate': DATA_DIR / f"gate_{TOKEN}_bbo.csv",
        'hype': DATA_DIR / f"hype_{TOKEN}_bbo.csv",
        'kusw': DATA_DIR / f"kusw_{TOKEN}_bbo.csv"
    }
    
    exchanges = list(bbo_paths.keys())
    pairs = list(itertools.combinations(exchanges, 2))
    
    signals, bbo_storage = {}, {}
    n_steps = len(df_master)
    timestamps = df_master.index
    
    def get_data_driven_epoch(exch_name, full_name):
        epoch_col = f'epoch_id_{full_name}'
        if epoch_col not in df_master.columns:
            epoch_col = f'epoch_id_{exch_name}'
        if epoch_col in df_master.columns:
            shift_mask = df_master[epoch_col] != df_master[epoch_col].shift(-1)
            shift_mask.iloc[-1] = False
            return shift_mask.values, shift_mask.sum()
        return np.zeros(n_steps, dtype=bool), 0

    for exch_a, exch_b in tqdm(pairs, desc="Pre-computing Dual Footprints"):
        pair_label = f"{exch_a.upper()}/{exch_b.upper()}"
        full_name_A = [k for k, v in NAME_MAP.items() if v == exch_a][0]
        full_name_B = [k for k, v in NAME_MAP.items() if v == exch_b][0]
        
        adj_col = f'spread_padj_idx_mid_avg_{full_name_A}_vs_{full_name_B}'
        raw_col = f'spread_raw_mid_{full_name_A}_vs_{full_name_B}'
        
        if adj_col not in df_master.columns or raw_col not in df_master.columns: continue
            
        bbo_df = load_aligned_bbo(df_master, exch_a, exch_b, bbo_paths)
        s_score, naive_z, raw_ma = calc_stochastic_ex_ante(df_master, adj_col, raw_col, ROLLING_WINDOW_MINS)
        z_exec_raw = df_master[raw_col].values
        
        mask_a = (timestamps.minute == 59) if exch_a == 'hype' else get_data_driven_epoch(exch_a, full_name_A)[0]
        mask_b = (timestamps.minute == 59) if exch_b == 'hype' else get_data_driven_epoch(exch_b, full_name_B)[0]
        is_funding_epoch = mask_a | mask_b

        signals[pair_label] = {
            's_score': s_score, 'naive_z': naive_z, 'raw_ma': raw_ma, 'raw_mid': z_exec_raw,
            'is_funding': is_funding_epoch,
            'liquidity_toll': bbo_df['liquidity_toll'].values,
            'round_trip_fee_bps': (FEES[exch_a] + FEES[exch_b]) * 2
        }
        bbo_storage[pair_label] = bbo_df

    sweep_results = []

    print("\n🔬 Commencing Head-to-Head Sensitivity Sweep...")
    
    for sig_type in ['naive', 'ou']:
        model_name = "Naive Raw Z-Score" if sig_type == 'naive' else "Normalized OU S-Score"
        for sig_thresh in tqdm(SIGMAS_TO_TEST, desc=f"Testing {model_name}"):
            tdf = run_engine_for_sigma_and_type(sig_thresh, sig_type, df_master, signals, bbo_storage, timestamps)
            
            if tdf.empty:
                sweep_results.append({'Model': model_name, 'Sigma': f"{sig_thresh}σ", 'Trades': 0, 'Win%': 0.0, 'Cum_PnL(bps)': 0.0, 'CalSR': 0.0, 'ActSR': 0.0, 'TrdSR': 0.0, 'MDD(bps)': 0.0, 'G2P': 0.0})
                continue
                
            tdf['close_date'] = pd.to_datetime(tdf['Exit_Time']).dt.date
            active_daily_bps = tdf.groupby('close_date')['Net_PnL'].sum()
            calendar_daily_bps = active_daily_bps.reindex(ALL_CALENDAR_DAYS, fill_value=0.0)
            
            global_cum_bps = calendar_daily_bps.sum()
            global_peaks = calendar_daily_bps.cumsum().cummax()
            global_mdd_bps = (global_peaks - calendar_daily_bps.cumsum()).max()
            win_pct = (tdf['Net_PnL'] > 0).mean() * 100
            
            cal_sharpe = (calendar_daily_bps.mean() / max(calendar_daily_bps.std(), 1e-6)) * np.sqrt(365)
            act_sharpe = (active_daily_bps.mean() / max(active_daily_bps.std(), 1e-6)) * np.sqrt(365) if len(active_daily_bps) > 1 else 0.0
            trd_sharpe = tdf['Net_PnL'].mean() / max(tdf['Net_PnL'].std(), 1e-6)
            
            gross_gains = tdf.loc[tdf['Net_PnL'] > 0, 'Net_PnL'].sum()
            gross_losses = abs(tdf.loc[tdf['Net_PnL'] < 0, 'Net_PnL'].sum())
            g2p_ratio = (gross_gains / gross_losses) if gross_losses > 0 else 999.99
            
            sweep_results.append({
                'Model': model_name,
                'Sigma': f"{sig_thresh}σ",
                'Trades': len(tdf),
                'Win%': round(win_pct, 2),
                'Cum_PnL(bps)': round(global_cum_bps, 2),
                'CalSR': round(cal_sharpe, 2),
                'ActSR': round(act_sharpe, 2),
                'TrdSR': round(trd_sharpe, 2),
                'MDD(bps)': round(global_mdd_bps, 2),
                'G2P': round(g2p_ratio, 2)
            })

    print("\n" + "="*125)
    print(" 📊 HEAD-TO-HEAD COMPARISON: NAIVE VS NORMALIZED ENGINE (UNCONSTRAINED BTC)")
    print("="*125)
    summary_df = pd.DataFrame(sweep_results)
    print(summary_df.to_string(index=False))
    print("="*125 + "\n")

if __name__ == "__main__":
    run_comparison()

🚀 Loading Tagged Master Parquet for BTC from:
   C:\Users\tz_ho\crypto-perpetual-arbitrage\data


Pre-computing Dual Footprints:   0%|          | 0/15 [00:00<?, ?it/s]


🔬 Commencing Head-to-Head Sensitivity Sweep...


Testing Naive Raw Z-Score:   0%|          | 0/3 [00:00<?, ?it/s]

Testing Normalized OU S-Score:   0%|          | 0/3 [00:00<?, ?it/s]


 📊 HEAD-TO-HEAD COMPARISON: NAIVE VS NORMALIZED ENGINE (UNCONSTRAINED BTC)
                Model Sigma  Trades  Win%  Cum_PnL(bps)  CalSR  ActSR  TrdSR  MDD(bps)  G2P
    Naive Raw Z-Score  3.0σ     333 23.12       -440.50  -1.47  -2.51  -0.10    741.69 0.64
    Naive Raw Z-Score  3.5σ     247 29.15       -125.75  -0.55  -1.02  -0.03    438.33 0.85
    Naive Raw Z-Score  4.0σ     181 37.57        265.95   1.19   2.37   0.09    139.57 1.53
Normalized OU S-Score  3.0σ     225 27.56       -104.39  -0.42  -0.81  -0.03    408.38 0.87
Normalized OU S-Score  3.5σ     162 36.42        247.17   1.15   2.40   0.08    133.09 1.52
Normalized OU S-Score  4.0σ     117 44.44        432.66   2.01   5.08   0.18     57.92 2.53

